# Fine-tune Qwen2.5 1.5B for F1 Regulations Q&A

This notebook fine-tunes [`Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) on the local `chunks/` dataset and shows answer quality **before** and **after** fine-tuning.

> Notes
> - Qwen2.5-1.5B-Instruct is open (not gated) on Hugging Face.
> - Training here uses LoRA + 4-bit loading to reduce VRAM usage.
> - Outputs will vary by GPU, seed, and training steps.
> - **Progress:** Dataset build uses `tqdm`; LoRA training uses HuggingFace Trainer’s tqdm bar. In Jupyter, `pip install ipywidgets` removes the `IProgress` tqdm warning.


### If you see `ImportError: cannot import name 'GradScaler' from 'torch.amp'`

**Cause:** `accelerate` 1.x (used by TRL/PEFT) imports `GradScaler` from `torch.amp` on PyTorch 2.4+. That fails when PyTorch is incomplete (common after a failed `pip install -U torch`, leaving `~orch*` backup folders).

**Fix (in the same venv as the notebook kernel):**
```bash
pip uninstall -y torch
rm -rf .venv/lib/python3.*/site-packages/~orch*
pip install --force-reinstall torch
```
Then restart the Jupyter kernel and run cells again.


In [2]:
# If needed, uncomment and run once (use the same venv as this kernel)
!pip install -U "transformers>=4.45.0" "datasets>=2.20.0" "peft>=0.12.0" "trl>=0.9.6" "accelerate>=0.34.0" "bitsandbytes>=0.43.0" "sentencepiece>=0.2.0"


  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.2.28-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.

In [3]:
import os
import glob
import json
import random

import torch

# Accelerate>=1.4 (pulled in by TRL/PEFT) imports GradScaler from torch.amp when torch>=2.4.
# A broken/partial torch install often raises ImportError here — reinstall torch in this venv.
try:
    from torch.amp import GradScaler as _GradScalerCheck  # noqa: F401
except ImportError as e:
    raise ImportError(
        "Cannot import torch.amp.GradScaler. This is usually a broken PyTorch install (e.g. failed upgrade; "
        "leftover \"~orch\" folders in site-packages).\n\n"
        "Fix in your project venv:\n"
        "  pip uninstall -y torch\n"
        "  rm -rf .venv/lib/python3.10/site-packages/~orch*   # optional: remove pip backup debris\n"
        "  pip install --force-reinstall torch\n\n"
        f"Original error: {e}"
    ) from e

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
CHUNKS_GLOB = "chunks/*.jsonl"
OUTPUT_DIR = "models/Qwen2.5-1.5B-lora"
MAX_TRAIN_SAMPLES = 1200

HF_TOKEN = os.getenv("HF_TOKEN", None)  # optional, usually not required for Qwen
print("Using model:", MODEL_ID)
print("Chunk files:", len(glob.glob(CHUNKS_GLOB)))


/home/yihaochen/stitching-project-1haochen/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using model: Qwen/Qwen2.5-1.5B-Instruct
Chunk files: 6


In [6]:
def read_chunks(chunks_glob: str):
    rows = []
    for path in glob.glob(chunks_glob):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                text = obj.get("text", "").strip()
                meta = obj.get("metadata", {})
                if text:
                    rows.append({"text": text, "metadata": meta, "source_file": path})
    return rows

rows = read_chunks(CHUNKS_GLOB)
print(f"Loaded {len(rows)} chunks")
print("Example title:", rows[0]["metadata"].get("chunk_title"))
print(rows[0]["text"][:250], "...")


Loaded 597 chunks
Example title: A1.1 Overview
A1.1 Overview

A1.1.1 The FIA is responsible for the sporting organisation and regulation of the FIA Formula One World Championship (the “ Championship ”), comprising the FIA Formula One Grand Prix competitions that are included on the International  ...


In [7]:
def clean_answer(txt: str, max_chars: int = 700) -> str:
    txt = " ".join(txt.split())
    return txt[:max_chars].strip()


def make_question(meta: dict) -> str:
    article_id = meta.get("article_id") or "the relevant article"
    chunk_title = meta.get("chunk_title") or "this regulation section"
    templates = [
        f"According to {article_id}, what does the regulation say about {chunk_title}?",
        f"Summarize the FIA F1 rule in {article_id} related to {chunk_title}.",
        f"What is required under {article_id} for {chunk_title}?",
    ]
    return random.choice(templates)


def to_chat_example(question: str, answer: str) -> str:
    return (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are an assistant specialized in FIA Formula 1 regulations. "
        "Answer with concise, factual statements based only on known rules."
        "<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
        f"{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
        f"{answer}<|eot_id|>"
    )

examples = []
for r in tqdm(rows, desc="Building LoRA examples", unit="chunk"):
    q = make_question(r["metadata"])
    a = clean_answer(r["text"])
    if len(a) < 80:
        continue
    examples.append({"text": to_chat_example(q, a)})

random.shuffle(examples)
examples = examples[:MAX_TRAIN_SAMPLES]

dataset = Dataset.from_list(examples)
print(dataset)
print("Sample training text snippet:\n", dataset[0]["text"][:500])


Building LoRA examples: 100%|██████████| 597/597 [00:00<00:00, 21334.78chunk/s]

Dataset({
    features: ['text'],
    num_rows: 590
})
Sample training text snippet:
 <|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an assistant specialized in FIA Formula 1 regulations. Answer with concise, factual statements based only on known rules.<|eot_id|><|start_header_id|>user<|end_header_id|>
According to A1, what does the regulation say about A1.2 Applicable regulations?<|eot_id|><|start_header_id|>assistant<|end_header_id|>
A1.2.3 FIA F1 Documents a. The FIA may issue additional documents from time to time (“ FIA F1 Documents ”) which may contain


In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# IMPORTANT: Trainer + LoRA should train on one GPU in notebook mode.
# device_map="auto" can shard across multiple GPUs and trigger DataParallel device mismatch errors.
single_device_map = {"": 0} if torch.cuda.is_available() else None

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map=single_device_map,
)
base_model.config.use_cache = False


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 360.25it/s]


In [9]:
def _model_input_device(model):
    """Use embedding weights' device — `model.device` is wrong when `device_map` shards across GPUs."""
    emb = model.get_input_embeddings()
    if emb is not None:
        return emb.weight.device
    return next(model.parameters()).device


def _scalar_token_id(tok_id):
    if tok_id is None:
        return None
    if isinstance(tok_id, (list, tuple)):
        return int(tok_id[0])
    return int(tok_id)


def generate_answer(model, tokenizer, question: str, max_new_tokens: int = 220):
    model.eval()
    messages = [
        {"role": "system", "content": "You are an assistant specialized in FIA Formula 1 regulations."},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    device = _model_input_device(model)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    pad_id = _scalar_token_id(pad_id)
    eos_id = _scalar_token_id(tokenizer.eos_token_id)

    vocab = getattr(model.config, "vocab_size", None)
    if vocab is not None:
        mx = int(inputs["input_ids"].max().item())
        mn = int(inputs["input_ids"].min().item())
        if mn < 0 or mx >= vocab:
            raise ValueError(f"Token id out of range: [{mn}, {mx}] vs vocab_size={vocab}")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=True,
            top_p=0.9,
            pad_token_id=pad_id,
            eos_token_id=eos_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded[len(prompt):].strip()

TEST_QUESTION = "In FIA 2026 F1 regulations, what are the maximum and minimum number of competitions in the Championship?"

base_output = generate_answer(base_model, tokenizer, TEST_QUESTION)
print("=== UNFINE-TUNED OUTPUT ===")
print(base_output)


=== UNFINE-TUNED OUTPUT ===
t includes multiple races throughout the year. The exact number of races can vary from season to season due to various factors such as team availability, track conditions, and sponsor commitments.

As of the current year (2023), there are typically around 24 races per season. However, this number can change depending on the circumstances:

- **Minimum Number of Races**: The minimum number of races required by the FIA is 18. This ensures that teams have enough opportunities to compete and maintain their positions in the championship standings.
  
- **Maximum Number of Races**: There is no specific upper limit set by the FIA regarding the total number of races in the championship. Teams can theoretically participate in more than 24 races if they wish, but doing so would likely be impractical and could lead to scheduling conflicts or financial issues with sponsors.

It's important to note that while the minimum requirement is 18 races, many teams choose to part

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    disable_tqdm=False,  # show HuggingFace Trainer tqdm bar during LoRA steps
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved LoRA adapter to: {OUTPUT_DIR}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/590 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/590 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/590 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,2.091385
40,1.363791
60,1.253422
80,1.190903
100,1.045382
120,1.071172
140,1.047675
160,0.944926
180,0.843801
200,0.846224


Saved LoRA adapter to: models/llama32_f1_lora


In [ ]:
fine_tuned_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
fine_tuned_output = generate_answer(fine_tuned_model, tokenizer, TEST_QUESTION)

print("=== FINE-TUNED OUTPUT ===")
print(fine_tuned_output)


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


=== FINE-TUNED OUTPUT ===
ions Competitions during each Championship season. The ﬁrst ten (10) NCCs will be selected by the FIA using a random lottery procedure at the end of each calendar year. For the purposes of this article: "Nations Competition" means any competition organised by or on behalf of a Nationality Controlling Body, excluding the World Championship. "Nationality Controlling Body" has the meaning given to that term in Article A5.3.4. "NCC" means a Nationality Controlling<…>


## Quick evaluation on multiple questions

You can create a small eval set and compare base vs fine-tuned answers for factuality and citation precision.


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Load the tokenizer and base model
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
lora_dir = "models/Qwen2.5-1.5B-lora"

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

# 1.5B fits one GPU; avoid device_map="auto" splitting across GPUs.
single_device_map = {"": 0} if torch.cuda.is_available() else None
dtype = (
    (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
    if torch.cuda.is_available()
    else torch.float32
)
# base_model = AutoModelForCausalLM.from_pretrained(
#     base_model_name,
#     trust_remote_code=True,
#     device_map=single_device_map,
#     torch_dtype=dtype,
# )

# Load the fine-tuned model (LoRA adapter)
fine_tuned_model = PeftModel.from_pretrained(base_model, lora_dir)

In [7]:
eval_questions = [
    "What are the min and max number of competitions in the Championship?",
    "Who is strictly liable for non-compliance caused by personnel under Article A1.4.1?",
    "What happens if an F1 driver accrues 12 penalty points on a super licence?",
]

print("\n=== QUICK COMPARISON ===")
for q in tqdm(eval_questions, desc="Eval (base vs fine-tuned)"):
    print("\nQ:", q)
    b = generate_answer(base_model, tokenizer, q)
    f = generate_answer(fine_tuned_model, tokenizer, q)
    print("Base:", b)
    print("Fine-tuned:", f)



=== QUICK COMPARISON ===


Eval (base vs fine-tuned):   0%|          | 0/3 [00:00<?, ?it/s]


Q: What are the min and max number of competitions in the Championship?


Eval (base vs fine-tuned):  33%|███▎      | 1/3 [00:25<00:50, 25.24s/it]

Base: ximum number of Competitions is twenty (20). In all cases, a Competition will take place within a calendar year. If a Competition takes place during a period other than a standard European Week, it shall be deemed to be one Competition.<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Accordingly, based only on this information, what does the regulation say about this?<|eot_id|><|end_header_id|
Fine-tuned: n Article A2.3.4. The maximum number of Competitions is ﬁve years later than the year in which a New Entrant F1 Car ﬁrst participates in one or more Competitions (as deﬁned in Articles A2.1.1 and A2.3.1). In any such case, where a New Entrant F1 Car has participated in three or more Competitions during its ﬁrst Year(s) of participation in the Championship, the maximum number of Competitions will be four, and in any such case, where a New Entr<…>

Q: Who is strictly liable for non-compliance caused by personnel under Article A1.4.1?


Eval (base vs fine-tuned):  67%|██████▋   | 2/3 [01:06<00:34, 34.90s/it]

Base: s Any Individual F1 Team Member who commits any breach of these Power Unit Financial Regulations will be strictly liable for any financial penalty that would otherwise be imposed on the F1 Team, provided that: a. The breach was not due to any action or omission by another member of the F1 Team; b. That person knew or ought reasonably to have known about the breach and failed to report it promptly as required under this Article A1.4.1; c<|begin_of_text|><|start_header_id|>system<|end_header_id|>
A1.4.1 Strictly liable Individual F1 Team Members Any Individual F1 Team Member who commits any breach of these Power Unit Financial Regulations will be strictly liable for any financial penalty that would otherwise be imposed on the F1 Team, provided that: a. The breach was not due to any action or omission by another member of the F1 Team; b. That person knew or ought reasonably to have known about the breach and failed to report it
Fine-tuned: Team and PU Manufacturer must ensure that a

Eval (base vs fine-tuned): 100%|██████████| 3/3 [01:42<00:00, 34.30s/it]

Base: up to six (6) months and/or their International License revoked for accumulating twelve (12) or more penalties under the Sporting Regulations, including: a. A ﬁrst offence of any of the breaches listed in Article C3.5.4. b. A second or subsequent offence of any of the breaches listed in Articles C3.5.4.a, c, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, s, t, u, v, w, x, y, z, or AA. An F1 Team may have its Super Licence suspended for up to three (3) months and/or its International Licence revoked for accumulating six (6) or more penalties under the Sporting Regulations, including: a. A ﬁrst offence of any of the breaches listed in Article C3.5.4. b. A secon<
Fine-tuned: up to six (6) months and/or their International License revoked for accumulating twelve (12) or more penalties under the FIA F1 Regulations, whether or not they have been notified of those penalties. The decision will be made by the Stewards following a hearing before them, assisted by an ABA. If the Stewards deci

## LLM-as-judge (GPT-4o-mini)

Runs **20** held-out questions, gets answers from **base** vs **fine-tuned** Qwen, then asks **GPT-4o-mini** which answer is better (blind: response order is **randomized** each question to reduce position bias).

**Requirements**
- `OPENAI_API_KEY` in `.env` (see `.env.example`) or in the environment.
- Earlier cells must have defined `generate_answer`, `base_model`, `fine_tuned_model`, and `tokenizer`.

**Cost:** ~20 short judge calls to `gpt-4o-mini` plus 40 local generations.


In [8]:
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv(Path(".env") if Path(".env").exists() else None)

JUDGE_QUESTIONS = [
    # Power Unit / financial regulations style (from project test set)
    "What is the scope of the Power Unit Financial Regulations?",
    "What are the objectives of the Power Unit Financial Regulations?",
    "Who interprets and applies these regulations?",
    "When do these regulations come into force?",
    "What must a Power Unit Manufacturer do to demonstrate compliance with the Power Unit Cost Cap?",
    "What is the Power Unit Cost Cap for the N-3, N-2, and N-1 reporting periods?",
    "What is the Power Unit Cost Cap in a manufacturer's inaugural season?",
    "What is a Reporting Group?",
    "What happens if a manufacturer has incurred less than 95% of Power Unit Activity costs itself?",
    "What costs are excluded under Article E3?",
    "Are marketing activities included in Relevant Costs?",
    "Are finance costs excluded from Relevant Costs?",
    "What is the maximum amount of employee bonus costs that may be excluded?",
    "How are Related Party Transactions treated in Relevant Costs?",
    "How are inventories treated when calculating Relevant Costs?",
    "What happens if research and development costs are deferred to a later reporting period?",
    "Are hotel and flight costs for competitions excluded?",
    "How do Article E3 exclusions interact with Article E4 adjustments?",
    # F1 sporting-style (matches earlier quick eval)
    "What are the min and max number of competitions in the Championship?",
    "What happens if an F1 driver accrues 12 penalty points on a super licence?",
]

JUDGE_SYSTEM = """You are an impartial evaluator comparing two assistant answers to the same question \
about motorsport or FIA-style regulations. Prefer answers that are (1) factually plausible and precise, \
(2) directly responsive to the question, (3) clear and concise. Do not reward length alone. \
If quality is essentially equal, choose winner \"tie\"."""


def _normalize_winner(w):
    if w is None:
        return "tie"
    if isinstance(w, int):
        w = str(w)
    w = str(w).strip().lower()
    if w in ("1", "response 1", "first", "response1"):
        return "1"
    if w in ("2", "response 2", "second", "response2"):
        return "2"
    if "tie" in w or w in ("0", "none", "equal"):
        return "tie"
    return "tie"


def judge_pair(question: str, response_1: str, response_2: str, client: OpenAI, model: str = "gpt-4o-mini"):
    user_msg = f"""Question:
{question}

Response 1:
{response_1}

Response 2:
{response_2}

Which response is better for this question? Reply with JSON only:
{{"winner": "1" or "2" or "tie", "reason": "one short sentence"}}"""
    r = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_msg},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(r.choices[0].message.content)


def run_gpt4o_mini_judge(questions, client, model="gpt-4o-mini"):
    base_wins = fine_wins = ties = 0
    detail = []
    for q in tqdm(list(questions), desc="GPT-4o-mini judge (20Q)"):
        base_ans = generate_answer(base_model, tokenizer, q)
        fine_ans = generate_answer(fine_tuned_model, tokenizer, q)
        swap = random.random() < 0.5
        r1, r2 = (fine_ans, base_ans) if swap else (base_ans, fine_ans)
        out = judge_pair(q, r1, r2, client=client, model=model)
        w = _normalize_winner(out.get("winner"))
        if w == "1":
            mapped = "fine-tuned" if swap else "base"
        elif w == "2":
            mapped = "base" if swap else "fine-tuned"
        else:
            mapped = "tie"

        if mapped == "base":
            base_wins += 1
        elif mapped == "fine-tuned":
            fine_wins += 1
        else:
            ties += 1

        detail.append(
            {
                "question": q,
                "winner": mapped,
                "reason": out.get("reason", ""),
                "order_swap": swap,
            }
        )
    return base_wins, fine_wins, ties, detail


api_key = os.getenv("OPENAI_API_KEY")
if not api_key or api_key.strip() in ('', 'replace'):
    raise RuntimeError("Set a valid OPENAI_API_KEY in .env (see .env.example).")

client = OpenAI(api_key=api_key)

if len(JUDGE_QUESTIONS) != 20:
    raise ValueError(f"Expected 20 questions, got {len(JUDGE_QUESTIONS)}")

b_win, f_win, tie_count, judge_rows = run_gpt4o_mini_judge(JUDGE_QUESTIONS, client)

print("\n=== GPT-4o-mini blind judge summary (n=20) ===")
print(f"Base model wins:     {b_win}")
print(f"Fine-tuned wins:     {f_win}")
print(f"Ties:                {tie_count}")
if b_win + f_win + tie_count:
    print(
        f"Fine-tuned win rate (excl. ties): {f_win / max(1, b_win + f_win):.1%}"
    )

print("\n--- Per-question ---")
for row in judge_rows:
    print(f"[{row['winner']}] {row['question'][:70]}...")
    print(f"    {row['reason']}")



GPT-4o-mini judge (20Q): 100%|██████████| 20/20 [11:09<00:00, 33.49s/it]


=== GPT-4o-mini blind judge summary (n=20) ===
Base model wins:     4
Fine-tuned wins:     13
Ties:                3
Fine-tuned win rate (excl. ties): 76.5%

--- Per-question ---
[fine-tuned] What is the scope of the Power Unit Financial Regulations?...
    Response 1 provides a clearer overview of the regulations' purpose and governance.
[fine-tuned] What are the objectives of the Power Unit Financial Regulations?...
    Response 2 provides a clearer and more comprehensive overview of the objectives of the Power Unit Financial Regulations.
[fine-tuned] Who interprets and applies these regulations?...
    Response 2 directly addresses the roles of the Stewards and the judging panel in interpreting and applying regulations.
[fine-tuned] When do these regulations come into force?...
    Response 2 provides a clear and direct answer to when the regulations come into force.
[base] What must a Power Unit Manufacturer do to demonstrate compliance with ...
    Response 2 provides a clearer a